# 🇲🇼 Chichewa Named Entity Recognition
## Fine-tuning DistilmBERT on Masakhane NER data

This notebook fine-tunes a multilingual language model to identify named entities in Chichewa text.

**What is Named Entity Recognition (NER)?**
NER is the task of finding and classifying named things in text — people, organisations, locations, and dates. For example:

> *"**Bingu wa Mutharika** anali mtsogoleri wa **Malawi** kuchokera mu **2004**."*
> - `Bingu wa Mutharika` → **PERSON**
> - `Malawi` → **LOCATION**  
> - `2004` → **DATE**

**What we use:**
- **Model:** `distilbert-base-multilingual-cased` — a compact multilingual model that understands 104 languages including Chichewa
- **Data:** Masakhane NER v2 — a dataset of African language text with named entity labels, created by the Masakhane research community

**Session plan:**
1. Set up and download everything (Cell 1)
2. Explore the dataset (Cell 2)
3. Prepare the data for training (Cell 3)
4. Fine-tune the model — 1 epoch on a small subset (Cell 4)
5. Evaluate and run inference (Cell 5)
6. Run inference with a fully trained model (Cell 6)

---

**Install dependencies (run once in terminal if needed):**
```bash
pip install transformers datasets seqeval torch
```

## Cell 1 — Setup & Download

This cell downloads everything we need and saves it locally so the rest of the notebook runs without internet.

**What gets downloaded:**
- The Masakhane NER v2 dataset for Chichewa (`nya` = Nyanja/Chichewa) — about 5MB
- The DistilmBERT model weights — about 135MB, cached in `~/.cache/huggingface`

**Run this cell once.** After it completes, you can run the rest of the notebook offline.

> **What is `nya`?** It's the ISO 639-3 language code for Chichewa (also called Nyanja). Masakhane uses this code to identify the Chichewa subset of their dataset.

In [ ]:
# ── CELL 1: Setup & Download ──────────────────────────────────────────────

import os
from datasets import load_dataset
from transformers import AutoTokenizer

# ── Configuration ─────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-multilingual-cased"  # ~135MB, fast on CPU
DATASET_NAME = "masakhane/masakhaner2"              # Masakhane NER v2
LANGUAGE = "nya"                                    # Chichewa (Nyanja)
SAVE_DIR = "./ner_model"                            # Where to save the trained model
DATA_DIR = "./ner_data"                             # Where to save the dataset

# ── Download dataset ──────────────────────────────────────────────────────
print("Downloading Masakhane Chichewa NER dataset...")
dataset = load_dataset(DATASET_NAME, LANGUAGE)
print(f"Done. Splits: {list(dataset.keys())}")
print(f"  Train: {len(dataset['train'])} sentences")
print(f"  Validation: {len(dataset['validation'])} sentences")
print(f"  Test: {len(dataset['test'])} sentences")

# Save dataset locally so it works offline
os.makedirs(DATA_DIR, exist_ok=True)
dataset.save_to_disk(DATA_DIR)
print(f"\nDataset saved to {DATA_DIR}/")

# ── Download tokenizer ────────────────────────────────────────────────────
print(f"\nDownloading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained("./tokenizer_cache")
print("Tokenizer saved to ./tokenizer_cache/")

# ── Download model weights ────────────────────────────────────────────────
# We download the base model now so it's cached for Cell 4
print(f"\nDownloading model weights for {MODEL_NAME} (~135MB)...")
from transformers import AutoModel
AutoModel.from_pretrained(MODEL_NAME)
print("Model weights cached.")

print("\n✅ Setup complete — everything is now available offline.")

## Cell 2 — Explore the Dataset

Before training anything, it's important to understand the data. This cell loads the saved dataset and shows you what it looks like.

**NER label format — BIO tagging:**
The dataset uses BIO (Beginning, Inside, Outside) tags:
- `O` — not a named entity
- `B-PER` — Beginning of a person name
- `I-PER` — Inside a person name (continuation)
- `B-ORG` — Beginning of an organisation
- `B-LOC` — Beginning of a location
- `B-DATE` — Beginning of a date

So for a two-word person name like `Bingu Mutharika`, the first word gets `B-PER` and the second gets `I-PER`.

**Why BIO?** It lets us handle multi-word entities. Without it, we couldn't tell whether two adjacent person names are one entity or two.

In [ ]:
# ── CELL 2: Explore the Dataset ───────────────────────────────────────────

from datasets import load_from_disk

# Load from local disk (no internet needed after Cell 1)
dataset = load_from_disk(DATA_DIR)

# Get the label names from the dataset features
label_names = dataset["train"].features["ner_tags"].feature.names
print("NER labels:", label_names)
print(f"Total label types: {len(label_names)}")

# Show a few examples
print("\n" + "─" * 60)
print("Sample sentences from the training set:")
print("─" * 60)

for i in range(3):
    example = dataset["train"][i]
    tokens = example["tokens"]
    tags = [label_names[t] for t in example["ner_tags"]]
    
    print(f"\nExample {i + 1}:")
    print(f"  Sentence: {' '.join(tokens)}")
    print(f"  Tokens:   {tokens}")
    print(f"  NER tags: {tags}")
    
    # Highlight named entities
    entities = []
    current = []
    current_type = None
    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current:
                entities.append((" ".join(current), current_type))
            current = [token]
            current_type = tag[2:]
        elif tag.startswith("I-") and current:
            current.append(token)
        else:
            if current:
                entities.append((" ".join(current), current_type))
            current = []
            current_type = None
    if current:
        entities.append((" ".join(current), current_type))
    
    if entities:
        print(f"  Entities found: {entities}")
    else:
        print(f"  Entities found: none")

# Label distribution
print("\n" + "─" * 60)
print("Label distribution in training set:")
from collections import Counter
all_tags = [label_names[t] for ex in dataset["train"] for t in ex["ner_tags"]]
for tag, count in sorted(Counter(all_tags).items(), key=lambda x: -x[1]):
    print(f"  {tag:<12} {count:>6} tokens")

## Cell 3 — Tokenization & Data Preparation

Before we can train the model, we need to convert the raw text into numbers the model understands. This is called **tokenization**.

**The key challenge — subword tokenization:**
BERT-style models split words into subword pieces. For example:
- `Mutharika` might become `['Mut', '##har', '##ika']`

But our NER labels are at the word level — one label per word. So when a word splits into 3 pieces, we need to decide what label each piece gets.

**Our solution:** Give the first subword piece the original label, and mark the rest with `-100` (a special value PyTorch ignores during training). This way the model learns from the first piece and ignores the rest.

**What is a tokenizer?** It's a vocabulary lookup — it converts text into token IDs (numbers) that the model can process. `distilbert-base-multilingual-cased` has a vocabulary of 119,547 tokens covering 104 languages.

In [ ]:
# ── CELL 3: Tokenization & Data Preparation ───────────────────────────────

from transformers import AutoTokenizer
from datasets import load_from_disk

dataset = load_from_disk(DATA_DIR)
label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

# Load tokenizer from local cache
tokenizer = AutoTokenizer.from_pretrained("./tokenizer_cache")

# ── Show what tokenization does ───────────────────────────────────────────
example_sentence = dataset["train"][0]["tokens"]
print("Original words:", example_sentence)
encoding = tokenizer(example_sentence, is_split_into_words=True)
print("Token IDs:", encoding["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoding["input_ids"]))
print("Word IDs:", encoding.word_ids())  # maps each token back to its word

# ── Tokenization function ─────────────────────────────────────────────────
def tokenize_and_align_labels(examples):
    """
    Tokenizes words into subword pieces and aligns NER labels.
    - First subword of each word gets the original label
    - Subsequent subwords get -100 (ignored during training)
    - Special tokens [CLS] and [SEP] also get -100
    """
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,  # tells tokenizer these are already split words
        max_length=128,            # cap sequence length for memory efficiency
    )
    
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        aligned_labels = []
        
        for word_id in word_ids:
            if word_id is None:
                # Special token ([CLS], [SEP]) — ignore
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word — use the real label
                aligned_labels.append(labels[word_id])
            else:
                # Continuation subword — ignore
                aligned_labels.append(-100)
            previous_word_id = word_id
        
        all_labels.append(aligned_labels)
    
    tokenized["labels"] = all_labels
    return tokenized

# ── Apply tokenization to full dataset ───────────────────────────────────
print("\nTokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Done.")
print(f"Training examples: {len(tokenized_dataset['train'])}")
print(f"Columns after tokenization: {tokenized_dataset['train'].column_names}")

## Cell 4 — Fine-tune the Model (Live Demo)

This is the training cell. We take DistilmBERT — a model already trained on 104 languages — and fine-tune it specifically for Chichewa NER.

**What is fine-tuning?**
The base model already understands language structure from pre-training on Wikipedia and other text. Fine-tuning adds a small classification layer on top and teaches it to predict NER labels for Chichewa specifically. We only need a small amount of data and a short training time because we're building on top of existing language knowledge.

**Training parameters:**
- `TRAIN_SUBSET` — how many examples to use (200 for the live demo, increase for better results)
- `NUM_EPOCHS` — how many passes through the data (1 for the demo)
- `LEARNING_RATE` — how fast the model updates (2e-5 is standard for fine-tuning BERT models)
- `BATCH_SIZE` — how many examples to process at once (8 is safe for CPU with 16GB RAM)

**Expected time on CPU:**
- 200 examples, 1 epoch ≈ 5–10 minutes
- Full dataset (~700 examples), 3 epochs ≈ 60–90 minutes

In [ ]:
# ── CELL 4: Fine-tune the Model ───────────────────────────────────────────

# ── Training configuration — change these to experiment ──────────────────
TRAIN_SUBSET = 200    # number of training examples (use None for full dataset)
NUM_EPOCHS = 1        # training passes (increase for better results)
LEARNING_RATE = 2e-5  # standard for BERT fine-tuning
BATCH_SIZE = 8        # examples per batch (safe for CPU with 16GB RAM)

# ── Imports ───────────────────────────────────────────────────────────────
import numpy as np
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

# ── Create label mappings ─────────────────────────────────────────────────
# The model needs to know which number corresponds to which label
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}
print("Label mapping:", id2label)

# ── Load model ────────────────────────────────────────────────────────────
# AutoModelForTokenClassification adds a classification head on top of DistilmBERT
# The head outputs one score per label for each token
print(f"\nLoading {MODEL_NAME}...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # expected — we're adding a new classification head
)
print(f"Model loaded. Parameters: {model.num_parameters():,}")

# ── Subset the training data for the live demo ────────────────────────────
train_data = tokenized_dataset["train"]
if TRAIN_SUBSET:
    train_data = train_data.select(range(min(TRAIN_SUBSET, len(train_data))))
    print(f"\nUsing {len(train_data)} training examples (subset for demo)")
else:
    print(f"\nUsing full training set: {len(train_data)} examples")

# ── Data collator ─────────────────────────────────────────────────────────
# Pads sequences in each batch to the same length
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# ── Training arguments ────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,          # regularisation to prevent overfitting
    eval_strategy="epoch",      # evaluate after each epoch
    save_strategy="epoch",      # save checkpoint after each epoch
    load_best_model_at_end=True,
    logging_steps=10,           # print loss every 10 steps
    use_cpu=True,               # force CPU (no GPU in Codespaces)
    report_to="none",           # don't send metrics anywhere
)

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ── Train ─────────────────────────────────────────────────────────────────
print(f"\nStarting training: {NUM_EPOCHS} epoch(s) on {len(train_data)} examples...")
print("Watch the loss decrease as the model learns.\n")
trainer.train()

# ── Save the fine-tuned model ─────────────────────────────────────────────
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\n✅ Model saved to {SAVE_DIR}/")

## Cell 5 — Evaluate & Run Inference

Now we test the model we just trained.

**Evaluation metrics for NER:**
- **Precision** — of all the entities the model predicted, what fraction were correct?
- **Recall** — of all the real entities in the text, what fraction did the model find?
- **F1 score** — the harmonic mean of precision and recall. This is the main metric. 1.0 is perfect, 0.0 is useless. After 1 epoch on 200 examples, expect 0.2–0.5.

**Inference** means running the trained model on new text to make predictions. We load the saved model and pass it Chichewa sentences to see what entities it finds.

> **Note:** After training on only 200 examples for 1 epoch, the model will make mistakes. That's expected and is part of what we're showing — you can see exactly how more data and more epochs improve results.

In [ ]:
# ── CELL 5: Evaluate & Run Inference ──────────────────────────────────────

from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
import numpy as np

# ── Load the fine-tuned model ─────────────────────────────────────────────
print("Loading fine-tuned model...")
ft_model = AutoModelForTokenClassification.from_pretrained(SAVE_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

# ── Evaluate on test set ──────────────────────────────────────────────────
print("\nEvaluating on test set...")
results = trainer.evaluate(tokenized_dataset["test"])
print(f"\nTest set results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# ── Run inference on sample sentences ────────────────────────────────────
# The pipeline handles tokenization and prediction automatically
ner_pipeline = pipeline(
    "ner",
    model=ft_model,
    tokenizer=ft_tokenizer,
    aggregation_strategy="simple",  # merges B- and I- tags into one entity
    device=-1,                       # -1 = CPU
)

# Sample Chichewa sentences — add your own here
test_sentences = [
    "Bingu wa Mutharika anali mtsogoleri wa Malawi.",
    "Lilongwe ndi mzinda waukulu wa Malawi.",
    "Boma la Malawi linagwirizana ndi dziko la Tanzania mu 2019.",
    "Aphunzitsi a sukulu ya Zomba akuphunzitsa ana.",
]

print("\n" + "─" * 60)
print("Inference on sample Chichewa sentences:")
print("─" * 60)

for sentence in test_sentences:
    print(f"\nSentence: {sentence}")
    predictions = ner_pipeline(sentence)
    if predictions:
        for ent in predictions:
            print(f"  → '{ent['word']}' = {ent['entity_group']} (score: {ent['score']:.3f})")
    else:
        print("  → No entities found")

print("\n" + "─" * 60)
print("Try your own sentence:")
my_sentence = "Lowani dzina lanu pano"  # replace with your own Chichewa sentence
predictions = ner_pipeline(my_sentence)
print(f"Sentence: {my_sentence}")
if predictions:
    for ent in predictions:
        print(f"  → '{ent['word']}' = {ent['entity_group']} (score: {ent['score']:.3f})")
else:
    print("  → No entities found")

## Cell 6 — Experiment: Change the Training Parameters

Now it's your turn. Go back to **Cell 4** and change these values, then run Cells 4 and 5 again:

| Parameter | Demo value | Try this |
|-----------|------------|----------|
| `TRAIN_SUBSET` | 200 | 500 or `None` (full dataset) |
| `NUM_EPOCHS` | 1 | 3 or 5 |
| `LEARNING_RATE` | 2e-5 | 5e-5 (faster) or 1e-5 (slower, more careful) |

**Questions to explore:**
- Does the F1 score improve with more training data?
- What happens if you train for more epochs?
- Which entity type (PER, ORG, LOC, DATE) does the model find most reliably?

**Understanding the loss:**
During training you see a loss number printed every 10 steps. This measures how wrong the model's predictions are. As training progresses, the loss should decrease — that means the model is learning.

---

## What next?

Once you have a well-trained model you can:
- Run it on the newspaper articles extracted in the other notebooks to find people, places, and organisations mentioned in TiKAMBE
- Push it to HuggingFace Hub to share it: `trainer.push_to_hub("your-username/chichewa-ner")`
- Use it as a base for other Chichewa NLP tasks

In [ ]:
# ── CELL 6: Quick summary of what we did ─────────────────────────────────

print("═" * 60)
print("TRAINING SUMMARY")
print("═" * 60)
print(f"Model:          {MODEL_NAME}")
print(f"Dataset:        Masakhane NER v2 — Chichewa (nya)")
print(f"Training examples used: {len(train_data)}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Model saved to: {SAVE_DIR}/")
print("═" * 60)
print("\nTo use this model on a new sentence:")
print('''
from transformers import pipeline
ner = pipeline("ner", model="./ner_model", aggregation_strategy="simple")
ner("Lowani dzina lanu mzinda wa Lilongwe.")
''')